# Import Libraries

In [ ]:
import nltk 
import re 
import ast
import html
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
%pip install python-docx
from docx import Document
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer 
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer 

# Load all Datasets

In [ ]:
animes = pd.read_csv("animes.csv")
profiles = pd.read_csv("profiles.csv")
reviews = pd.read_csv("reviews.csv")

# Data Description and Relationships
#### Inspect the source datasets, their schemas, and potential join keys before cleaning.

In [ ]:
datasets = {
    'Animes': animes,
    'Profiles': profiles,
    'Reviews': reviews
}

for name, df in datasets.items():
    print(f"\n{'='*40}")
    print(f"{name + ' Dataset':^40}")
    print(f"{'='*40}")

    print("\nFirst 5 Rows:")
    display(df.head())
    
    print("\nDataset Information:")
    df.info()

    print("\nDescriptive Statistics:")
    display(df.describe(include='all'))

    #print("\nShape:", df.shape)

    print("\nMissing Values:")
    print(df.isnull().sum())

    print("\nDuplicate Rows:")
    print(df.duplicated().sum())

    

# Data Cleaning and Preparation
#### Remove duplicate records and assess missing or inconsistent values across the three sources.

In [ ]:
# Duplicate handling

cleaning_summary = []

def clean_duplicates(df, name, key):
    original = len(df)
    exact = int(df.duplicated().sum())
    no_exact = df.drop_duplicates().copy()
    repeated_ids = int(no_exact.duplicated(subset=[key]).sum())
    cleaned = no_exact.drop_duplicates(subset=[key], keep="first").copy()
    cleaning_summary.append({
        "Dataset": name, "Original Rows": original,
        "Exact Duplicates Removed": exact,
        "Additional Duplicate IDs Removed": repeated_ids,
        "Cleaned Rows": len(cleaned)
    })
    return cleaned

animes_clean = clean_duplicates(animes, "Anime", "uid")
profiles_clean = clean_duplicates(profiles, "Profiles", "profile")
reviews_clean = clean_duplicates(reviews, "Reviews", "uid")

cleaning_summary_df = pd.DataFrame(cleaning_summary)
display(cleaning_summary_df)
print("Cleaned shapes:", animes_clean.shape, profiles_clean.shape, reviews_clean.shape)

In [ ]:
# Missing Values and Numeric Consistency

for col in ["uid", "episodes", "members", "popularity", "ranked", "score"]:
    animes_clean[col] = pd.to_numeric(animes_clean[col], errors="coerce")
for col in ["uid", "anime_uid", "score"]:
    reviews_clean[col] = pd.to_numeric(reviews_clean[col], errors="coerce")

print("Anime missing values after conversion:")
display(animes_clean.isna().sum().to_frame("Missing"))
print("Reviews missing values after conversion:")
display(reviews_clean.isna().sum().to_frame("Missing"))

# Text Preprocessing
#### Prepare review text for analysis. English stopwords are loaded form NLTK.

In [ ]:
# Download English stopwords
nltk.download('stopwords')

# Load English stopwords
stop_words = set(stopwords.words('english'))

print(f"Total stopwords loaded: {len(stop_words)}")

stop_words.add("etc")

# Preserve negation words
stop_words -= {"no", "not", "nor", "never"}


def preprocess(text):

    if pd.isna(text):
        return ""

    # Convert HTML entities
    text = html.unescape(str(text))

    # Remove anime review rating headers
    text = re.sub(
        r'^\s*(?:more\s+pics\s*)?Overall\s*\d+\s*'
        r'Story\s*\d+\s*Animation\s*\d+\s*'
        r'Sound\s*\d+\s*Character\s*\d+\s*'
        r'Enjoyment\s*\d+',
        '',
        text,
        flags=re.IGNORECASE
    )

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)

    # Convert to lowercase
    text = text.lower()

    # Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', ' ', text)

    # Remove stopwords
    tokens = [
        word for word in text.split()
        if word not in stop_words
    ]

    return " ".join(tokens)


# Apply preprocessing
reviews_clean["processed_text"] = (
    reviews_clean["text"].apply(preprocess)
)

# Display results
display(
    reviews_clean[["text", "processed_text"]].head(3)
)
    

# Feature Engineering
#### Create anime-level metrics from source attributes, review aggregates, release information, and profile favorites.

In [ ]:
# Function to convert genre strings into lists
def parse_genres(value):
    if pd.isna(value) or not str(value).strip():
        return []

    try:
        result = ast.literal_eval(str(value))
        return result if isinstance(result, list) else []
    except (ValueError, SyntaxError):
        return []

In [ ]:
# Create the anime-level feature table
anime_features = animes_clean.copy()

# Review features: one row per anime ID
review_summary = (
    reviews_clean.groupby("anime_uid", dropna=True)
    .agg(review_count=("uid", "count"), avg_review_score=("score", "mean"))
    .reset_index()
)

# Popularity index: lower original rank is better
min_rank = anime_features["popularity"].min()
max_rank = anime_features["popularity"].max()
if pd.notna(min_rank) and pd.notna(max_rank) and max_rank > min_rank:
    anime_features["popularity_index"] = (
        (max_rank - anime_features["popularity"]) / (max_rank - min_rank) * 100
    )
else:
    anime_features["popularity_index"] = np.nan

# Genre count
anime_features["genre_count"] = anime_features["genre"].apply(
    lambda value: len(parse_genres(value))
)

# Heuristic franchise flag: title mentions sequels or later installments
franchise_pattern = r"\b(?:season\s*\d+|\d+(?:st|nd|rd|th)\s+season|part\s*\d+|sequel)\b"
anime_features["franchise_indicator"] = (
    anime_features["title"].fillna("")
    .str.contains(franchise_pattern, case=False, regex=True)
    .astype(int)
)

# Episode count is already present: numeric conversion was performed above.

# Merge review aggregates into anime; preserve all anime rows
final_dataset = anime_features.merge(
    review_summary, left_on="uid", right_on="anime_uid",
    how="left", validate="one_to_one"
).drop(columns=["anime_uid"])
final_dataset["review_count"] = final_dataset["review_count"].fillna(0).astype(int)
# avg_review_score remains NaN when there are no matching reviews.
print("Anime + review features:", final_dataset.shape)
display(final_dataset[["title", "score", "episodes", "popularity_index", "genre_count",
                       "franchise_indicator", "review_count", "avg_review_score"]].head())

In [ ]:
# =====================================================
# RELEASE YEAR AND ANIME AGE FEATURE ENGINEERING
# =====================================================
# Extract release year and anime age


# Extract the first four-digit year from aired
animes_clean["release_year"] = pd.to_numeric(
    animes_clean["aired"].astype(str).str.extract(
        r"\b((?:19|20)\d{2})\b",
        expand=False
    ),
    errors="coerce"
).astype("Int64")

# Calculate anime age using 2026
animes_clean["anime_age"] = (
    2026 - animes_clean["release_year"]
).astype("Int64")

# Group anime by release decade
animes_clean["release_decade"] = (
    (animes_clean["release_year"] // 10) * 10
).astype("Int64")

# Add features to the final merged dataset using anime UID
release_features = animes_clean[
    ["uid", "release_year", "anime_age", "release_decade"]
].drop_duplicates(subset="uid")

# Remove existing versions if rerunning this cell
final_dataset = final_dataset.drop(
    columns=["release_year", "anime_age", "release_decade"],
    errors="ignore"
)

final_dataset = final_dataset.merge(
    release_features,
    on="uid",
    how="left",
    validate="one_to_one"
)

# Preview results
display(
    final_dataset[
        ["title", "aired", "release_year",
         "anime_age", "release_decade"]
    ].head(10)
)

print("Missing release years:",
      final_dataset["release_year"].isna().sum())

print("Final dataset shape:", final_dataset.shape)

In [ ]:
# Profile favorites feature

# favorites_anime contains strings such as "['33352', '25013']"
def parse_favorites(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(str(value))
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError, TypeError):
        return []

profiles_fav = profiles_clean[["profile", "favorites_anime"]].copy()
profiles_fav["favorites_anime"] = profiles_fav["favorites_anime"].apply(parse_favorites)
favorites = profiles_fav.explode("favorites_anime")
favorites["favorites_anime"] = pd.to_numeric(favorites["favorites_anime"], errors="coerce")
favorites = favorites.dropna(subset=["favorites_anime", "profile"])
favorites = favorites.drop_duplicates(subset=["profile", "favorites_anime"])

favorites_summary = (
    favorites.groupby("favorites_anime")["profile"]
    .nunique().reset_index(name="favorites_count")
)

final_dataset = final_dataset.merge(
    favorites_summary, left_on="uid", right_on="favorites_anime",
    how="left", validate="one_to_one"
).drop(columns=["favorites_anime"])
final_dataset["favorites_count"] = final_dataset["favorites_count"].fillna(0).astype(int)

print("Final dataset shape:", final_dataset.shape)
print("Duplicate anime IDs:", final_dataset["uid"].duplicated().sum())
print("Total matched profile–anime favorites:", final_dataset["favorites_count"].sum())
display(final_dataset.head())

## Data Quality Assessment

In [ ]:
# =====================================================
# FINAL DATA QUALITY ASSESSMENT
# =====================================================


df = final_dataset.copy()

# Missing values, including text placeholders
missing_counts = df.isna().sum().astype(int)

placeholders = {
    "", "not available", "unknown", "n/a", "none", "null"
}

for col in df.select_dtypes(include=["object", "string"]):
    placeholder_mask = (
        df[col].astype("string")
        .str.strip()
        .str.lower()
        .isin(placeholders)
        .fillna(False)
    )
    missing_counts[col] += int(placeholder_mask.sum())

completeness_report = pd.DataFrame({
    "Column": df.columns,
    "Missing Count": missing_counts.values,
    "Missing %": (missing_counts.values / len(df) * 100).round(2),
    "Completeness %": (
        100 - missing_counts.values / len(df) * 100
    ).round(2)
})

# Numeric validity checks
validity_checks = {
    "Score outside 0-10":
        int((df["score"].notna() &
             ~df["score"].between(0, 10)).sum()),
    "Negative episodes":
        int((df["episodes"] < 0).sum()),
    "Negative members":
        int((df["members"] < 0).sum()),
    "Invalid popularity rank":
        int((df["popularity"] < 1).sum()),
    "Popularity index outside 0-100":
        int((df["popularity_index"].notna() &
             ~df["popularity_index"].between(0, 100)).sum()),
    "Negative review counts":
        int((df["review_count"] < 0).sum()),
    "Average review score outside 0-10":
        int((df["avg_review_score"].notna() &
             ~df["avg_review_score"].between(0, 10)).sum()),
    "Negative favorites":
        int((df["favorites_count"] < 0).sum()),
    "Negative anime age":
        int((df["anime_age"] < 0).sum())
}

validity_report = pd.DataFrame({
    "Validation Check": list(validity_checks),
    "Invalid Records": list(validity_checks.values())
})

# Feature consistency checks
consistency_checks = {
    "Release decade mismatch": int((
        df["release_year"].notna() &
        (df["release_decade"] !=
         (df["release_year"] // 10) * 10)
    ).fillna(False).sum()),

    "Anime age mismatch": int((
        df["release_year"].notna() &
        (df["anime_age"] != 2026 - df["release_year"])
    ).fillna(False).sum()),

    "Review score without reviews": int((
        (df["review_count"] == 0) &
        df["avg_review_score"].notna()
    ).sum()),

    "Derived dates without release year": int((
        df["release_year"].isna() &
        (df["anime_age"].notna() |
         df["release_decade"].notna())
    ).sum())
}

consistency_report = pd.DataFrame({
    "Check": list(consistency_checks),
    "Count": list(consistency_checks.values())
})

overall_completeness = (
    (1 - missing_counts.sum() / df.size) * 100
)

quality_summary = pd.DataFrame({
    "Metric": [
        "Total records",
        "Total columns",
        "Overall completeness (%)",
        "Duplicate rows",
        "Duplicate anime IDs",
        "Missing anime IDs",
        "Invalid numeric values",
        "Consistency issues"
    ],
    "Result": [
        len(df),
        df.shape[1],
        round(overall_completeness, 2),
        int(df.duplicated().sum()),
        int(df["uid"].duplicated().sum()),
        int(df["uid"].isna().sum()),
        int(validity_report["Invalid Records"].sum()),
        int(consistency_report["Count"].sum())
    ]
})

print("DATA QUALITY ASSESSMENT")
display(quality_summary)

print("\nCOMPLETENESS")
display(completeness_report)

print("\nVALIDITY")
display(validity_report)

print("\nCONSISTENCY")
display(consistency_report)

# Data Dictionary
#### Export the merged dataset, cleaned source tables, data dictionary, and cleaning report.

In [ ]:
# 1. Create output folder
OUTPUT_DIR = Path("Anime_Project_Output2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Generate data dictionary from final dataset
definitions = {
    "uid": "Unique anime identifier",
    "title": "Anime title",
    "synopsis": "Anime description",
    "genre": "Anime genre labels",
    "aired": "Release or broadcast information",
    "episodes": "Number of episodes",
    "members": "Number of anime community members",
    "popularity": "Popularity ranking; lower is better",
    "ranked": "Overall anime ranking; lower is better",
    "score": "Overall anime rating",
    "img_url": "Anime image URL",
    "link": "Anime information URL",
    "popularity_index": "Normalized popularity score (0-100)",
    "genre_count": "Number of genres assigned",
    "franchise_indicator": "Possible related-release title flag",
    "review_count": "Number of reviews for each anime",
    "avg_review_score": "Average valid review score",
    "review_score_std": "Variation in review scores",
    "favorites_count": "Number of profiles favoriting the anime",
    "release_year": "First release year extracted from the aired column",
    "anime_age": "Years since first release, calculated as of 2026",
    "release_decade": "Decade of first release, used for historical trend analysis"
}

data_dictionary = pd.DataFrame({
    "Column": final_dataset.columns,
    "Data Type": [
        str(final_dataset[c].dtype)
        for c in final_dataset.columns
    ],
    "Description": [
        definitions.get(c, "Description needed")
        for c in final_dataset.columns
    ],
    "Missing Values": [
        final_dataset[c].isna().sum()
        for c in final_dataset.columns
    ]
})

# 3. Save final merged and clean datasets
final_dataset.to_csv(
    OUTPUT_DIR / "final_anime_dataset.csv",
    index=False
)
# Export the three cleaned datasets
animes_clean.to_csv(
    OUTPUT_DIR / "anime_clean.csv",
    index=False
)

profiles_clean.to_csv(
    OUTPUT_DIR / "profiles_clean.csv",
    index=False
)

reviews_clean.to_csv(
    OUTPUT_DIR / "reviews_clean.csv",
    index=False
)

# 4. Save data dictionary as Word document
doc = Document()
doc.add_heading("Anime Acquisition - Data Dictionary", 0)

table = doc.add_table(rows=1, cols=4)
table.style = "Table Grid"

for i, heading in enumerate(data_dictionary.columns):
    table.rows[0].cells[i].text = heading

for _, row in data_dictionary.iterrows():
    cells = table.add_row().cells
    for i, value in enumerate(row):
        cells[i].text = str(value)

doc.save(OUTPUT_DIR / "anime_data_dictionary.docx")

# 5. Calculate current dataset statistics
anime_rows = len(animes_clean)
review_rows = len(reviews_clean)
profile_rows = len(profiles_clean)

final_rows, final_cols = final_dataset.shape
duplicate_ids = final_dataset["uid"].duplicated().sum()

favorites_total = (
    int(final_dataset["favorites_count"].sum())
    if "favorites_count" in final_dataset.columns else 0
)

# 6. Calculate current validation statistics
final_rows, final_cols = final_dataset.shape
duplicate_ids = int(final_dataset["uid"].duplicated().sum())

missing_years = int(
    final_dataset["release_year"].isna().sum()
)

valid_years = final_dataset["release_year"].dropna()

earliest_year = int(valid_years.min())
latest_year = int(valid_years.max())

valid_year_count = len(valid_years)
valid_year_pct = valid_year_count / final_rows * 100

favorites_total = int(
    final_dataset["favorites_count"].sum()
)


# ==========================================================
# 7. GENERATE UPDATED CLEANING REPORT
# ==========================================================

# Calculate release-date statistics
release_years = pd.to_numeric(
    final_dataset["release_year"], errors="coerce"
)

valid_years = release_years.dropna()
missing_years = int(release_years.isna().sum())

earliest_year = int(valid_years.min()) if len(valid_years) else "N/A"
latest_year = int(valid_years.max()) if len(valid_years) else "N/A"

valid_year_count = len(valid_years)
valid_year_pct = (
    valid_year_count / len(final_dataset) * 100
    if len(final_dataset) else 0
)

# Calculate missing values, including text placeholders
placeholders = {
    "", "not available", "unknown", "n/a", "none", "null"
}

missing_counts = final_dataset.isna().sum().astype(int)

for col in final_dataset.select_dtypes(
    include=["object", "string"]
).columns:
    placeholder_mask = (
        final_dataset[col].astype("string")
        .str.strip()
        .str.lower()
        .isin(placeholders)
        .fillna(False)
    )
    missing_counts[col] += int(placeholder_mask.sum())

# Calculate overall completeness
total_cells = final_dataset.size
overall_completeness = (
    (1 - missing_counts.sum() / total_cells) * 100
    if total_cells else 0
)

# Data quality validation
duplicate_rows = int(final_dataset.duplicated().sum())
missing_ids = int(final_dataset["uid"].isna().sum())

# These tables are created in the Data Quality Assessment cell
invalid_numeric = int(
    validity_report["Invalid Records"].sum()
)

consistency_issues = int(
    consistency_report["Count"].sum()
)

# Generate report
report = f"""
ANIME ACQUISITION - DATA CLEANING REPORT

Cleaned datasets:
Anime: {anime_rows:,}
Reviews: {review_rows:,}
Profiles: {profile_rows:,}

Cleaning actions:
- Removed duplicate records and identifiers.
- Reviewed missing values and numerical data types.
- Checked scores and episode counts for invalid values.
- Parsed genre and favorite anime lists.
- Created popularity index and genre counts.
- Created exploratory franchise indicators.
- Calculated review counts and average review scores.
- Calculated distinct profile-anime favorite counts.
- Merged the three datasets into one anime-level dataset.
- Retained potential outliers for further investigation.

Release date cleaning and feature engineering:
- Examined the aired column for inconsistent date formats.
- Identified {missing_years:,} records with unavailable release years.
- Extracted release_year from the aired column.
- Created anime_age using 2026 as the reference year.
- Created release_decade for historical comparisons.
- Preserved missing release dates without estimation.

Release date validation:
Earliest release year: {earliest_year}
Latest release year: {latest_year}
Valid release years: {valid_year_count:,} ({valid_year_pct:.2f}%)
Missing release years: {missing_years:,}

Data quality assessment:
Overall completeness: {overall_completeness:.2f}%
Duplicate records: {duplicate_rows:,}
Duplicate anime IDs: {duplicate_ids:,}
Missing anime IDs: {missing_ids:,}
Invalid numeric values in tested ranges: {invalid_numeric:,}
Feature consistency issues detected: {consistency_issues:,}

Validation:
Matched favorite relationships: {favorites_total:,}
Duplicate anime IDs: {duplicate_ids:,}

Final dataset:
Rows: {final_rows:,}
Columns: {final_cols:,}

Remaining missing values:
"""

# Add missing values dynamically
for col, count in missing_counts.items():
    if count > 0:
        percentage = count / final_rows * 100
        report += f"- {col}: {count:,} ({percentage:.2f}%)\n"

report += """
Investment relevance:
- Release year supports comparisons across release periods.
- Anime age helps identify established titles.
- Release decade supports historical trend analysis.
- Ratings and membership provide historical audience signals.
- Review-based features require caution due to missing data.

Limitations:
- Data may represent different collection snapshots.
- The dataset contains release years only through 2021.
- Favorites cover only the available user profiles.
- Some anime have no available review scores.
- Popularity index is relative to this dataset.
- Franchise indicators are unverified title-based estimates.
- Extreme episode counts and membership values were retained.
- Licensing costs and ownership rights are unavailable.
- Historical audience metrics may not reflect current demand.
"""

# ==========================================================
# 8. SAVE CLEANING REPORT AS WORD DOCUMENT
# ==========================================================

report_doc = Document()

report_doc.add_heading(
    "Anime Acquisition - Data Cleaning Report", 0
)

for line in report.strip().splitlines():
    line = line.strip()

    if not line:
        continue

    # Avoid repeating the document title
    if line == "ANIME ACQUISITION - DATA CLEANING REPORT":
        continue

    if line.endswith(":"):
        report_doc.add_heading(line, level=2)

    elif line.startswith("- "):
        report_doc.add_paragraph(
            line[2:], style="List Bullet"
        )

    else:
        report_doc.add_paragraph(line)

report_doc.save(
    OUTPUT_DIR / "anime_cleaning_report.docx"
)

# Save Notepad version
with open(
    OUTPUT_DIR / "anime_cleaning_report.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(report.strip() + "\n")

# ==========================================================
# 9. EXPORT DATA QUALITY AND DUPLICATE SUMMARIES
# ==========================================================

# Export data quality results
quality_summary.to_csv(
    OUTPUT_DIR / "data_quality_summary.csv",
    index=False
)

completeness_report.to_csv(
    OUTPUT_DIR / "data_quality_completeness.csv",
    index=False
)

validity_report.to_csv(
    OUTPUT_DIR / "data_quality_validity.csv",
    index=False
)

consistency_report.to_csv(
    OUTPUT_DIR / "data_quality_consistency.csv",
    index=False
)

# Export duplicate cleaning summary if available
if "cleaning_summary_df" in globals():
    cleaning_summary_df.to_csv(
        OUTPUT_DIR / "duplicate_cleaning_summary.csv",
        index=False
    )

# ==========================================================
# 10. CONFIRM ALL EXPORTED FILES
# ==========================================================

print("ALL PROJECT DELIVERABLES EXPORTED SUCCESSFULLY!")
print("\nSaved in:", OUTPUT_DIR.resolve())

print(f"\nFinal dataset: {final_rows:,} rows x {final_cols} columns")
print(f"Overall completeness: {overall_completeness:.2f}%")
print(f"Missing release years: {missing_years:,}")

print("\nExported files:")
for file in sorted(OUTPUT_DIR.iterdir()):
    print("-", file.name)